# LipophilicityPrediction Batch Inference

This notebook has two parts: first, check whether the current kernel can use the GPU; second, batch-predict CSV files with a single `SMILES` column. Each file is processed and written immediately as `*_processed.csv`.

In [1]:
# ===== Absolute path configuration =====
REPO_DIR = "/home/cenking/VsCode/LipophilicityPrediction"
INPUT_DIR = "/home/cenking/VsCode/Data_Port/Candidate"
OUTPUT_DIR = "/home/cenking/VsCode/Data_Port/LipophilicityPrediction"

# Use the fold_0 checkpoint you currently have. If you later collect all folds,
# you can set CHECKPOINT_PATH = None and CHECKPOINT_DIR to the exp_0 folder.
CHECKPOINT_PATH = "/home/cenking/VsCode/LipophilicityPrediction/data/raw/baselines/dmpnn/logs/exp_0/fold_0/model_0/model.pt"
CHECKPOINT_DIR = None

# Inference settings. Set USE_GPU = False to force CPU.
USE_GPU = True
GPU_ID = 0
BATCH_SIZE = 50
NUM_WORKERS = 0
OVERWRITE = True


## 1. Kernel / GPU Check

In [2]:
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR).expanduser().resolve()
dmpnn_dir = repo_dir / "scripts" / "SOTA" / "dmpnn"

if not repo_dir.is_dir():
    raise FileNotFoundError(f"REPO_DIR does not exist: {repo_dir}")
if not (dmpnn_dir / "predict.py").is_file():
    raise FileNotFoundError(f"predict.py was not found: {dmpnn_dir / 'predict.py'}")

if str(dmpnn_dir) not in sys.path:
    sys.path.insert(0, str(dmpnn_dir))

import torch

print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    x = torch.tensor([1.0, 2.0], device=f"cuda:{GPU_ID}")
    print(f"GPU test tensor sum: {x.sum().item()}")
elif USE_GPU:
    print("USE_GPU=True, but CUDA is not available in this kernel. Prediction will fail unless USE_GPU is set to False.")


Python executable: /home/cenking/miniconda3/envs/LipophilicityPrediction/bin/python
PyTorch version: 2.10.0+cu130
CUDA available: True
CUDA device count: 1
GPU 0: NVIDIA GeForce RTX 5070 Laptop GPU
GPU test tensor sum: 3.0


## 2. Batch Prediction

In [3]:
from pathlib import Path
import subprocess
import pandas as pd
from tqdm.notebook import tqdm

input_dir = Path(INPUT_DIR).expanduser().resolve()
output_dir = Path(OUTPUT_DIR).expanduser().resolve()
checkpoint_path = Path(CHECKPOINT_PATH).expanduser().resolve() if CHECKPOINT_PATH else None
checkpoint_dir = Path(CHECKPOINT_DIR).expanduser().resolve() if CHECKPOINT_DIR else None

if not input_dir.is_dir():
    raise FileNotFoundError(f"INPUT_DIR does not exist or is not a directory: {input_dir}")
if checkpoint_path is None and checkpoint_dir is None:
    raise ValueError("Set either CHECKPOINT_PATH or CHECKPOINT_DIR.")
if checkpoint_path is not None and not checkpoint_path.is_file():
    raise FileNotFoundError(f"CHECKPOINT_PATH does not exist: {checkpoint_path}")
if checkpoint_dir is not None and not checkpoint_dir.is_dir():
    raise FileNotFoundError(f"CHECKPOINT_DIR does not exist: {checkpoint_dir}")

output_dir.mkdir(parents=True, exist_ok=True)
csv_files = sorted(input_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in INPUT_DIR: {input_dir}")

print(f"Input files: {len(csv_files)}")
print(f"Output directory: {output_dir}")
print(f"Checkpoint path: {checkpoint_path}")
print(f"Checkpoint dir: {checkpoint_dir}")


Input files: 1
Output directory: /home/cenking/VsCode/Data_Port/LipophilicityPrediction
Checkpoint path: /home/cenking/VsCode/LipophilicityPrediction/data/raw/baselines/dmpnn/logs/exp_0/fold_0/model_0/model.pt
Checkpoint dir: None


In [4]:
def validate_input_csv(csv_path: Path) -> None:
    header = pd.read_csv(csv_path, nrows=0)
    if list(header.columns) != ["SMILES"]:
        raise ValueError(f"{csv_path} must contain exactly one column named 'SMILES'. Found: {list(header.columns)}")


def output_path_for(csv_path: Path) -> Path:
    return output_dir / f"{csv_path.stem}_processed.csv"


def build_predict_command(input_csv: Path, output_csv: Path) -> list[str]:
    cmd = [
        sys.executable,
        str(dmpnn_dir / "predict.py"),
        "--test_path", str(input_csv),
        "--preds_path", str(output_csv),
        "--smiles_column", "SMILES",
        "--features_generator", "rdkit_wo_fragments_and_counts",
        "--no_features_scaling",
        "--batch_size", str(BATCH_SIZE),
        "--num_workers", str(NUM_WORKERS),
    ]

    if checkpoint_path is not None:
        cmd += ["--checkpoint_path", str(checkpoint_path)]
    else:
        cmd += ["--checkpoint_dir", str(checkpoint_dir)]

    if USE_GPU:
        cmd += ["--gpu", str(GPU_ID)]
    else:
        cmd += ["--no_cuda"]

    return cmd


def process_one_file(csv_path: Path) -> Path:
    validate_input_csv(csv_path)
    out_path = output_path_for(csv_path)

    if out_path.exists() and not OVERWRITE:
        return out_path

    tmp_path = out_path.with_suffix(".tmp.csv")
    if tmp_path.exists():
        tmp_path.unlink()

    cmd = build_predict_command(csv_path, tmp_path)
    result = subprocess.run(
        cmd,
        cwd=str(dmpnn_dir),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if result.returncode != 0:
        if tmp_path.exists():
            tmp_path.unlink()
        raise RuntimeError(
            f"Prediction failed for {csv_path}\n"
            f"Command: {' '.join(cmd)}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    tmp_path.replace(out_path)
    return out_path


In [5]:
written_files = []

for csv_path in tqdm(csv_files, desc="Processed files", unit="file"):
    written_files.append(process_one_file(csv_path))

print(f"Done. Wrote {len(written_files)} files to {output_dir}")
written_files[:5]


Processed files:   0%|          | 0/1 [00:00<?, ?file/s]

Done. Wrote 1 files to /home/cenking/VsCode/Data_Port/LipophilicityPrediction


[PosixPath('/home/cenking/VsCode/Data_Port/LipophilicityPrediction/candidate_only_SMILES_processed.csv')]